In [2]:
import os
# Project structure
PROJECT_ROOT = os.path.dirname(os.getcwd())
FACE_RECOGNITION_PATH = os.path.join(PROJECT_ROOT, 'app', 'models', 'face_recognition')
FACE_RECOGNITION_MODEL = os.path.join(FACE_RECOGNITION_PATH, 'arc.onnx')

In [3]:
import onnxruntime as ort
import numpy as np
import cv2

In [4]:
# Load ArcFace ONNX model
session = ort.InferenceSession(FACE_RECOGNITION_MODEL, providers=["CPUExecutionProvider"])

In [5]:
inp = session.get_inputs()[0]
print("Input name:", inp.name)
print("Shape:", inp.shape)
print("Type:", inp.type)

Input name: input_1
Shape: ['unk__556', 112, 112, 3]
Type: tensor(float)


In [ ]:
# Preprocessing for ArcFace
def preprocess(path):
    img = cv2.imread(path)
    if img is None:
        raise ValueError("Image not found:", path)

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (112, 112))

    img = img.astype(np.float32)
    img = img / 255.0       # normalize to 0–1 (most NHWC exports expect this)

    img = np.expand_dims(img, axis=0)   # (1,112,112,3)
    return img


# Cosine similarity
def cosine_similarity(a, b):
    a = a / np.linalg.norm(a)
    b = b / np.linalg.norm(b)
    return float(np.dot(a, b))

In [ ]:
inp_name = session.get_inputs()[0].name
print("Input name:", inp_name)

In [ ]:
# Pick two images
img1 = preprocess("ayush1.jpg")
img2 = preprocess("ayush2.jpg")

# Run inference
emb1 = session.run(None, {inp_name: img1})[0].flatten()
emb2 = session.run(None, {inp_name: img2})[0].flatten()

# Compute cosine similarity
sim = cosine_similarity(emb1, emb2)
print("Cosine Similarity:", sim)


In [ ]:
def test_pair(img_a, img_b):
    a = preprocess(img_a)
    b = preprocess(img_b)

    e1 = session.run(None, {inp_name: a})[0].flatten()
    e2 = session.run(None, {inp_name: b})[0].flatten()

    print(img_a, img_b, "->", cosine_similarity(e1, e2))


# Example:
test_pair("ayush1.jpg", "ayush2.jpg")        # same person
test_pair("ayush1.jpg", "random_other.jpg")  # different person